## import

In [ ]:
import numpy as np
import os
from multiprocessing import Pool
from kyupy import verilog, Timers, logic, batchrange, cdiv
from kyupy.logic import unpackbits, packbits
from kyupy.logic_sim import LogicSim
from kyupy.techlib import NANGATE45

### 1. 判定ロジック部 (確率的遅延対応版)

In [ ]:
def worker_sim_transition_fault(task_info, data):
  
    line_idx, fault_type = task_info
    a_curr, b_curr, s_curr, golden_r = data
    
    # 遷移故障モデルの定義に従い、キャプチャサイクルにおいて
    # 前の値から「遷移できなかった」状態を Stuck-at で表現する。
    if fault_type == 0: 
        sa_model = 0 # Slow-to-Rise -> SA0 として振る舞う
    else: 
        sa_model = 1 # Slow-to-Fall -> SA1 として振る舞う

    # 故障シミュレーション実行
    faulty_r = fma.sim(a_curr, b_curr, s_curr, fault_line=line_idx, fault_model=sa_model)
    
    # 正常値(Golden)と異なれば、遷移故障として検出可能
    is_detected = np.any(faulty_r != golden_r)

    return (fault_type, 1 if is_detected else 0)

### 2. シミュレータクラス定義 

In [ ]:
class FMA_Fixed(LogicSim):
    def __init__(self, netlist_file, sims=8):
        assert sims > 0 and sims % 8 == 0
        circuit = verilog.load(netlist_file, branchforks=False, tlib=NANGATE45)
        circuit.resolve_tlib_cells(NANGATE45)
        super().__init__(circuit, sims, m=2, c_reuse=False, strip_forks=False)

        self.r_locs = self.circuit.io_locs("o_sum")
        self.bit_width = len(self.r_locs) # 24ビット等
        
        # データ型の自動選択
        if self.bit_width <= 8:
            self.data_format = np.int8
        elif self.bit_width <= 16:
            self.data_format = np.int16
        elif self.bit_width <= 32:
            self.data_format = np.int32
        else:
            self.data_format = np.int64
            
        self.a_locs = self.circuit.io_locs("i_weight")
        self.b_locs = self.circuit.io_locs("i_activation")
        self.s_locs = self.circuit.io_locs("i_sum")
        
        self.s[0] = 0
        self.cycle()

    def _setup_input(self, a, b, s):
        nbytes = cdiv(len(a), 8)
        
        a_bits = np.ascontiguousarray(unpackbits(a).T[:len(self.a_locs)])
        b_bits = np.ascontiguousarray(unpackbits(b).T[:len(self.b_locs)])
        s_bits = np.ascontiguousarray(unpackbits(s).T[:len(self.s_locs)])
        
        self.s[0, self.a_locs, 0, :nbytes] = np.packbits(a_bits, axis=-1)
        self.s[0, self.b_locs, 0, :nbytes] = np.packbits(b_bits, axis=-1)
        self.s[0, self.s_locs, 0, :nbytes] = np.packbits(s_bits, axis=-1)

    def sim(self, a, b, s, fault_line=-1, fault_model=2):
        a, b, s = np.broadcast_arrays(a, b, s)
        a, b, s = [x.astype(self.data_format) for x in (a, b, s)]
        args = [x.flatten() for x in (a, b, s)]
        r = np.zeros_like(args[0])
        for bo, bs in batchrange(len(r), self.sims):
            r[bo:bo+bs] = self._sim(*[x[bo:bo+bs] for x in args], fault_line, fault_model)[:bs]
        return r.reshape(a.shape)

    def _sim(self, a, b, s, fault_line=-1, fault_model=2):
        self._setup_input(a, b, s)
        self.s_to_c()
        self.c_prop(fault_line=int(fault_line), fault_model=fault_model)
        self.c_to_s()
        
        nbytes = cdiv(len(a), 8)
        rb = np.unpackbits(self.s[1, self.r_locs, 0, :nbytes], axis=-1)
        
        target_bits = np.dtype(self.data_format).itemsize * 8
        rb_padded = np.zeros((target_bits, rb.shape[1]), dtype=np.uint8)
        rb_padded[:len(self.r_locs), :] = rb
        
        r = packbits(rb_padded.T, dtype=self.data_format)
        return r

### メイン実行部

In [ ]:
if __name__ == "__main__":
    netlist_path = os.path.join('circuits', 'fma_fixed_8_24.gl.v')
    
    if not os.path.exists(netlist_path):
        print(f"Error: Netlist not found at {netlist_path}")
    else:
        global fma
        fma = FMA_Fixed(netlist_path, sims=1024)
        print(f"Circuit Loaded: {netlist_path}")
        print(f"  - Output Width: {fma.bit_width} bits")
        print(f"  - Using Data Type: {fma.data_format.__name__}")

        test_count = 1024
        
        a_curr = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        b_curr = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        s_curr = np.random.randint(-2147483648, 2147483647, size=test_count, dtype=np.int32)
        
        print("Running Golden Simulation...")
        golden_r = fma.sim(a_curr, b_curr, s_curr)

        # ターゲット選定
        sample_size = 100
        all_lines = list(range(len(fma.circuit.lines)))
        targets = np.random.choice(all_lines, min(sample_size, len(all_lines)), replace=False)

        print(f"\nExperiment: Standard Transition Fault Simulation")
        print(f"  Target Lines: {sample_size} (STR and STF)")
        print("-" * 50)

        data_packet = (a_curr, b_curr, s_curr, golden_r)
        
        tasks = []
        for l in targets:
            tasks.append((l, 0)) # STR
            tasks.append((l, 1)) # STF

        # 並列実行
        with Pool(processes=8) as pool:
            results = pool.starmap(worker_sim_transition_fault, [(t, data_packet) for t in tasks])

        det_str = sum(1 for r in results if r[0] == 0 and r[1] > 0)
        det_stf = sum(1 for r in results if r[0] == 1 and r[1] > 0)
        
        rate_str = det_str / sample_size * 100
        rate_stf = det_stf / sample_size * 100
        total_rate = (det_str + det_stf) / (2 * sample_size) * 100
        
        print(f"Slow-to-Rise (STR) Detection: {rate_str:.1f}%")
        print(f"Slow-to-Fall (STF) Detection: {rate_stf:.1f}%")
        print(f"Total Transition Fault Coverage: {total_rate:.1f}%")
        print("-" * 50)